In [1]:
!pip install imagehash -q

import os
import glob
import hashlib
from pathlib import Path
from collections import defaultdict
from PIL import Image
import imagehash
from tqdm import tqdm

# 1. Define your dataset path
# Update this if your path is slightly different in the new notebook
DATA_DIR = '/kaggle/input/datasets/sabuktagin/tropical-flowers/Tropical Flower Dataset Seven Species from Bangladesh for Classification and Ecological Research/Flower Dataset/Flower Dataset'

# Gather all image paths
image_paths = []
for ext in ['**/*.jpg', '**/*.jpeg', '**/*.png']:
    image_paths.extend(glob.glob(os.path.join(DATA_DIR, ext), recursive=True))

print(f"Found {len(image_paths)} total images. Starting analysis...\n")

# 2. Check for Exact Duplicates (MD5)
print("🔍 STEP 1: Scanning for EXACT duplicates (MD5 Hash)...")
exact_hashes = defaultdict(list)
for path in tqdm(image_paths, desc="MD5 Hashing"):
    with open(path, 'rb') as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    exact_hashes[file_hash].append(path)

exact_dupes = {h: paths for h, paths in exact_hashes.items() if len(paths) > 1}
exact_dupe_count = sum(len(paths) - 1 for paths in exact_dupes.values())
print(f"⚠️ Found {exact_dupe_count} exact duplicate images.\n")

# 3. Check for Visual/Near Duplicates (pHash)
# This catches images that are the same but maybe resized, slightly cropped, or compressed differently
print("🔍 STEP 2: Scanning for VISUAL/NEAR duplicates (Perceptual Hash)...")
visual_hashes = defaultdict(list)
for path in tqdm(image_paths, desc="pHash Hashing"):
    try:
        img = Image.open(path).convert('RGB')
        img_hash = str(imagehash.phash(img))
        visual_hashes[img_hash].append(path)
    except Exception as e:
        pass

visual_dupes = {h: paths for h, paths in visual_hashes.items() if len(paths) > 1}
visual_dupe_count = sum(len(paths) - 1 for paths in visual_dupes.values())
print(f"⚠️ Found {visual_dupe_count} visual duplicate images.\n")

# 4. Summary & Data Leakage Report
print("="*60)
print("📊 DUPLICATE & DATA LEAKAGE REPORT")
print("="*60)

if visual_dupe_count > 0:
    print(f"Total red-flag images: {visual_dupe_count}")
    print("\nHow this causes Data Leakage & 100% Accuracy:")
    print("If you use a random 80/10/10 split on this dataset, these duplicate images")
    print("will be scattered across your Train, Validation, and Test sets.")
    print("The model will 'memorize' the image in the Training set, and when it sees the")
    print("exact same image in the Validation/Test set, it will guess it correctly with 100% confidence.")
    print("\nSample of duplicated files to verify manually:")
    
    # Print up to 3 examples for you to manually look at
    count = 0
    for h, paths in visual_dupes.items():
        if count >= 3: break
        print(f"\nDuplicate Set {count+1}:")
        for p in paths:
            print(f" -> {p}")
        count += 1
else:
    print("✅ No duplicates found! Your 100% accuracy is NOT due to data leakage.")
    print("The dataset is just very distinguishable, and your transfer learning models are highly effective.")

Found 4317 total images. Starting analysis...

🔍 STEP 1: Scanning for EXACT duplicates (MD5 Hash)...


MD5 Hashing: 100%|██████████| 4317/4317 [00:30<00:00, 143.80it/s]


⚠️ Found 5 exact duplicate images.

🔍 STEP 2: Scanning for VISUAL/NEAR duplicates (Perceptual Hash)...


pHash Hashing: 100%|██████████| 4317/4317 [06:27<00:00, 11.15it/s]

⚠️ Found 11 visual duplicate images.

📊 DUPLICATE & DATA LEAKAGE REPORT
Total red-flag images: 11

How this causes Data Leakage & 100% Accuracy:
If you use a random 80/10/10 split on this dataset, these duplicate images
will be scattered across your Train, Validation, and Test sets.
The model will 'memorize' the image in the Training set, and when it sees the
exact same image in the Validation/Test set, it will guess it correctly with 100% confidence.

Sample of duplicated files to verify manually:

Duplicate Set 1:
 -> /kaggle/input/datasets/sabuktagin/tropical-flowers/Tropical Flower Dataset Seven Species from Bangladesh for Classification and Ecological Research/Flower Dataset/Flower Dataset/Bougainvillea/IMG_20241104_144526_2.jpg
 -> /kaggle/input/datasets/sabuktagin/tropical-flowers/Tropical Flower Dataset Seven Species from Bangladesh for Classification and Ecological Research/Flower Dataset/Flower Dataset/Bougainvillea/IMG_20241104_144526_1.jpg

Duplicate Set 2:
 -> /kaggle/inpu